In [ ]:
# ============================================================
# COLLEGE NOTES ASSISTANT - RAG
# Google Colab | SINGLE CELL
# ============================================================

# -----------------------------
# 1. INSTALL REQUIRED PACKAGES
# -----------------------------
import subprocess
import sys

packages = [
    "pypdf",
    "sentence-transformers",
    "faiss-cpu",
    "transformers",
    "accelerate"
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + packages
)

# -----------------------------
# 2. IMPORT LIBRARIES
# -----------------------------
import os
import re
import numpy as np
import torch
import faiss

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import files

# -----------------------------
# 3. UPLOAD COLLEGE NOTES
# -----------------------------
print("=" * 70)
print("       📚 COLLEGE NOTES RAG ASSISTANT")
print("=" * 70)

print("\nUpload your PDF college notes.")
print("You can upload one or multiple PDF files.\n")

uploaded = files.upload()

pdf_files = [
    filename
    for filename in uploaded.keys()
    if filename.lower().endswith(".pdf")
]

if not pdf_files:
    raise Exception("No PDF files were uploaded.")

print("\nUploaded files:")
for filename in pdf_files:
    print("✓", filename)

# -----------------------------
# 4. EXTRACT TEXT FROM PDFs
# -----------------------------
print("\nExtracting text from PDFs...")

documents = []

for pdf_file in pdf_files:

    reader = PdfReader(pdf_file)

    for page_number, page in enumerate(reader.pages):

        try:
            text = page.extract_text()
        except:
            text = ""

        if text and text.strip():

            documents.append({
                "text": text,
                "source": pdf_file,
                "page": page_number + 1
            })

print(f"✓ Extracted {len(documents)} pages.")

if not documents:
    raise Exception(
        "No text could be extracted. "
        "Your PDF may contain scanned images."
    )

# -----------------------------
# 5. CLEAN TEXT
# -----------------------------
def clean_text(text):

    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()

# -----------------------------
# 6. CREATE TEXT CHUNKS
# -----------------------------
def create_chunks(text, chunk_size=700, overlap=100):

    text = clean_text(text)

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = []

for document in documents:

    page_chunks = create_chunks(
        document["text"]
    )

    for chunk in page_chunks:

        chunks.append({
            "text": chunk,
            "source": document["source"],
            "page": document["page"]
        })

print(f"✓ Created {len(chunks)} text chunks.")

# -----------------------------
# 7. LOAD EMBEDDING MODEL
# -----------------------------
print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# -----------------------------
# 8. CREATE EMBEDDINGS
# -----------------------------
print("Creating embeddings...")

texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

print("✓ Embeddings created.")

# -----------------------------
# 9. CREATE FAISS DATABASE
# -----------------------------
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print(f"✓ FAISS database created with {index.ntotal} vectors.")

# -----------------------------
# 10. RETRIEVAL FUNCTION
# -----------------------------
def retrieve(query, k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        min(k, len(chunks))
    )

    results = []

    for distance, idx in zip(
        distances[0],
        indices[0]
    ):

        if idx == -1:
            continue

        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "page": chunks[idx]["page"],
            "distance": float(distance)
        })

    return results

# -----------------------------
# 11. LOAD LLM
# -----------------------------
print("\nLoading language model...")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=(
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    ),
    device_map="auto"
)

print("✓ Language model loaded.")

# -----------------------------
# 12. CREATE RAG PROMPT
# -----------------------------
def create_prompt(question, retrieved_docs):

    context = ""

    for i, doc in enumerate(
        retrieved_docs,
        start=1
    ):

        context += f"""

SOURCE {i}
File: {doc['source']}
Page: {doc['page']}

{doc['text']}

--------------------------------
"""

    prompt = f"""
You are an intelligent College Notes Assistant.

Your job is to answer the student's question using ONLY
the information contained in the provided college notes.

IMPORTANT RULES:

1. Do not invent information.
2. If the answer is not present in the notes, say:
   "I could not find this information in the uploaded notes."
3. Explain concepts clearly for a college student.
4. Use bullet points when appropriate.
5. Give the source file and page number at the end.
6. Do not use outside knowledge.

QUESTION:
{question}

COLLEGE NOTES:
{context}

ANSWER:
"""

    return prompt

# -----------------------------
# 13. GENERATE ANSWER
# -----------------------------
def generate_answer(question, k=5):

    retrieved_docs = retrieve(
        question,
        k=k
    )

    prompt = create_prompt(
        question,
        retrieved_docs
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=6000
    )

    # Move tensors to model device
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.1
        )

    # Remove prompt tokens
    generated_tokens = output[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip(), retrieved_docs

# -----------------------------
# 14. START CHATBOT
# -----------------------------
print("\n")
print("=" * 70)
print("       🎓 COLLEGE NOTES ASSISTANT READY!")
print("=" * 70)

print("""
You can now ask questions about your uploaded notes.

Examples:

• What is normalization in DBMS?
• Explain TCP and UDP.
• What are the advantages of operating systems?
• Explain Unit 3 in simple words.
• Give me important points from this chapter.
• Create 5 questions from this topic.
• What is the difference between X and Y?

Type 'exit' to stop.
""")

# -----------------------------
# 15. CHAT LOOP
# -----------------------------
while True:

    question = input("\n🧑‍🎓 You: ").strip()

    if question.lower() in [
        "exit",
        "quit",
        "q"
    ]:

        print("\n👋 Thank you for using College Notes Assistant!")
        break

    if not question:
        continue

    print("\n🔍 Searching your notes...")

    try:

        answer, sources = generate_answer(
            question,
            k=5
        )

        print("\n🤖 Assistant:")
        print("-" * 70)
        print(answer)

        print("\n📚 SOURCES:")
        print("-" * 70)

        # Remove duplicate source/page combinations
        seen = set()

        for source in sources:

            key = (
                source["source"],
                source["page"]
            )

            if key not in seen:

                print(
                    f"📄 {source['source']} "
                    f"| Page {source['page']}"
                )

                seen.add(key)

        print("-" * 70)

    except Exception as e:

        print("\n❌ Error:", str(e))

       📚 COLLEGE NOTES RAG ASSISTANT

Upload your PDF college notes.
You can upload one or multiple PDF files.



Saving Deep Learning.pdf to Deep Learning.pdf

Uploaded files:
✓ Deep Learning.pdf

Extracting text from PDFs...
✓ Extracted 15 pages.
✓ Created 28 text chunks.

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✓ Language model loaded.


       🎓 COLLEGE NOTES ASSISTANT READY!

You can now ask questions about your uploaded notes.

Examples:

• What is normalization in DBMS?
• Explain TCP and UDP.
• What are the advantages of operating systems?
• Explain Unit 3 in simple words.
• Give me important points from this chapter.
• Create 5 questions from this topic.
• What is the difference between X and Y?

Type 'exit' to stop.


🔍 Searching your notes...

🤖 Assistant:
----------------------------------------------------------------------
Normalization in DBMS refers to the process of adjusting numerical values so that they fall within a specific range or scale. This technique helps in standardizing data across different datasets, making it easier to compare and analyze them effectively. In database management systems, normalization ensures that each column contains only one type of data and reduces redundancy by organizing data into tables with fewer columns and rows. The primary goal of normaliza